# Otimização Optuna

In [ ]:
import pandas as pd
from datetime import datetime
from src.Anomaly.Optimizer import AnomalyOptunaOptimizer
from src.Data.Processor import DataStreamProcessor

# Definição dos datasets
categorias = ['Consistência', 'Generalização', 'Adaptação', 'Recorrência']
tamanhos = ['25', '200','1000']

datasets = [f'data/15k/{cat}/{cat}_{tam}.csv' for cat in categorias for tam in tamanhos]

FEATURES = [
    'Max Packet Length',
    'Average Packet Size',
    'Fwd Packet Length Min',
    'Min Packet Length',
    'Fwd Packet Length Max',
    'Packet Length Mean',
    'Fwd Packet Length Mean',
    'Avg Fwd Segment Size',
    'min_seg_size_forward',
    'ACK Flag Count',
    'Flow Duration',
    'Fwd IAT Total',
    'Flow IAT Max',
    'Fwd IAT Max',
    'Flow IAT Std',
    'Fwd IAT Std',
    'Fwd IAT Mean',
    'Flow IAT Mean',
    'Total Length of Fwd Packets',
    'Subflow Fwd Bytes',
    'act_data_pkt_fwd',
    'Subflow Fwd Packets',
    'Total Fwd Packets',
    'Down/Up Ratio',
    'Init_Win_bytes_backward',
    'Total Length of Bwd Packets',
    'Subflow Bwd Bytes',
    'Flow IAT Min',
    'Bwd Packet Length Max',
    'URG Flag Count',
    'Bwd IAT Total',
    'Bwd Packets/s',
    'Init_Win_bytes_forward',
]

In [ ]:
id_execucao = datetime.now().strftime("%Y%m%d_%H%M")

modelos_para_executar = ['AE', 'HST', 'AIF']
estrategias_discretizacao = ['z_score', 'dinamic']
configuracoes_features = [('FullFeatures', None), ('33Features', FEATURES)]

estrategias_decisao = [
    "raw",
    "moving_average_w3",
    "moving_average_w5",
    "persistence_2_of_3",
    "persistence_3_of_5",
]

n_trials_teste = 50
n_runs_teste = 5
warmup_instances = 2000
window_evaluation = 100
fp_penalty_weight = 0.5

for dataset_path in datasets:
    nome_experimento = dataset_path.split("/")[-1].replace(".csv", "")
    df = pd.read_csv(dataset_path)

    for feat_name, selected_feats in configuracoes_features:

        processor = DataStreamProcessor(
            logging=False,
            selected_features=selected_feats
        )

        stream, targets, features = processor.create_stream(
            df=df,
            target_label_col="Label",
            binary_label=False,
            normalize_method="MinMaxScaler",
            threshold_var=None,
            threshold_corr=None,
            top_n_features=None,
            return_stream=True,
            extra_ignore_cols=[
                "Source IP",
                "Source Port",
                "Destination IP",
                "Destination Port",
                "Protocol",
                "Inbound",
            ],
            imputation_method="mediana",
        )

        for estrategia_threshold in estrategias_discretizacao:
            for estrategia_decisao in estrategias_decisao:
                for model_name in modelos_para_executar:
                   
                    optimizer = AnomalyOptunaOptimizer(
                        stream=stream,
                        n_trials=n_trials_teste,
                        discretization_threshold=estrategia_threshold,
                        decision_strategy=estrategia_decisao,
                        fp_penalty_weight=fp_penalty_weight,
                        target_names=targets,
                        n_runs=n_runs_teste,
                    )

                    best_params = optimizer.optimize(
                        model_name=model_name,
                        warmup_instances=warmup_instances,
                        experiment_name=nome_experimento,
                        num_features=len(features),
                        exec_id=id_execucao,
                        window_evaluation=window_evaluation,
                    )

# Execução Default

In [ ]:
from src.Anomaly.Pipeline import AnomalyExperimentRunner
from src.Anomaly.Models import get_anomaly_models
from src.Data.Processor import DataStreamProcessor
from datetime import datetime
import pandas as pd

categorias = ['Adaptação', 'Recorrência']
tamanhos = ['25', '200', '1000']

datasets = [f'data/15k/{cat}/{cat}_{tam}.csv' for cat in categorias for tam in tamanhos]

FEATURES = [
    'Max Packet Length',
    'Average Packet Size',
    'Fwd Packet Length Min',
    'Min Packet Length',
    'Fwd Packet Length Max',
    'Packet Length Mean',
    'Fwd Packet Length Mean',
    'Avg Fwd Segment Size',
    'min_seg_size_forward',
    'ACK Flag Count',
    'Flow Duration',
    'Fwd IAT Total',
    'Flow IAT Max',
    'Fwd IAT Max',
    'Flow IAT Std',
    'Fwd IAT Std',
    'Fwd IAT Mean',
    'Flow IAT Mean',
    'Total Length of Fwd Packets',
    'Subflow Fwd Bytes',
    'act_data_pkt_fwd',
    'Subflow Fwd Packets',
    'Total Fwd Packets',
    'Down/Up Ratio',
    'Init_Win_bytes_backward',
    'Total Length of Bwd Packets',
    'Subflow Bwd Bytes',
    'Flow IAT Min',
    'Bwd Packet Length Max',
    'URG Flag Count',
    'Bwd IAT Total',
    'Bwd Packets/s',
    'Init_Win_bytes_forward',
]

In [ ]:
%load_ext autoreload
%autoreload 2

from datetime import datetime
import pandas as pd

default_params = {
    'AIF': {
        'window_size': 256,
        'n_trees': 100,
        'height': None,
        'm_trees': 10,
        'weights': 0.5
    },
    'HST': {
        'window_size': 250,
        'number_of_trees': 25,
        'max_depth': 15,
        'anomaly_threshold': 0.50,
        'size_limit': 0.10
    },
    'AE': {
        'hidden_layer': 2,
        'learning_rate': 0.5,
        'threshold': 0.60
    }
}

default_thresholds = {
    'AIF': 0.50,
    'HST': 0.50,
    'AE': 0.60
}

modelos_para_executar = ['AE', 'HST', 'AIF']

configuracoes_features = [
    ('FullFeatures', None),
    ('33Features', FEATURES)
]

id_execucao = datetime.now().strftime("%Y%m%d_%H%M")

for dataset_path in datasets:
    nome_experimento = dataset_path.split('/')[-1].replace('.csv', '')

    df = pd.read_csv(dataset_path)

    for feat_name, selected_feats in configuracoes_features:

        processor = DataStreamProcessor(logging=False, selected_features=selected_feats)

        stream, targets, features = processor.create_stream(
            df=df,
            target_label_col='Label',
            binary_label=False,
            normalize_method="MinMaxScaler",
            threshold_var=None,
            threshold_corr=None,
            top_n_features=None,
            return_stream=True,
            extra_ignore_cols=[
                'Source IP',
                'Source Port',
                'Destination IP',
                'Destination Port',
                'Protocol',
                'Inbound'
            ],
            imputation_method='mediana'
        )

        for model_name in modelos_para_executar:
            current_params = default_params[model_name].copy()
            discretization_val = default_thresholds[model_name]

            kwargs = {
                f"{model_name.lower()}_params": current_params
            }

            algoritmos = get_anomaly_models(
                stream.get_schema(),
                selected_models=[model_name],
                **kwargs
            )

            runner = AnomalyExperimentRunner(
                target_names=targets,
                n_runs=5
            )

            runner.run_anomaly_evaluation(
                stream=stream,
                algorithms=algoritmos,
                window_evaluation=100,
                warmup_instances=2000,
                title=nome_experimento,
                discretization=discretization_val,
                algorithm_params=current_params,
                is_optimized=False,
                num_features=len(features),
                exec_id=id_execucao,
                decision_strategy='raw'
            )